In [1]:
%load_ext autoreload
%autoreload 2

# B6 (partie 2) — Auto-encodeur & détection d'anomalies

Reprend les jeux prétraités en [partie 1](b6_deep_learning_donnees.md) (`screw/*.bin.gz`) pour concevoir,
entraîner et évaluer un auto-encodeur convolutionnel de détection d'anomalies.

In [2]:
import gzip
from pathlib import Path

import numpy as np

DATA_DIR = Path('screw')
IMG_SIZE = 128


def load_bin_gz(path, img_size=IMG_SIZE):
    with gzip.open(path, 'rb') as f:
        data = np.frombuffer(f.read(), dtype='float32')
    return data.reshape(-1, img_size, img_size, 1)


X_train_norm = load_bin_gz(DATA_DIR / 'train_norm.bin.gz')
test_good_norm = load_bin_gz(DATA_DIR / 'test_good_norm.bin.gz')
test_defects_norm_stacked = load_bin_gz(DATA_DIR / 'test_defects_norm.bin.gz')

print('Train (normé)  :', X_train_norm.shape, X_train_norm.dtype)
print('Test saines    :', test_good_norm.shape)
print('Test défauts   :', test_defects_norm_stacked.shape)

Train (normé)  : (256, 128, 128, 1) float32
Test saines    : (41, 128, 128, 1)
Test défauts   : (119, 128, 128, 1)


Conception de l'auto-encodeur

In [3]:
from tensorflow import keras
from tensorflow.keras import layers

INPUT_SHAPE = X_train_norm.shape[1:]  # (128, 128, 1)

inputs = keras.Input(shape=INPUT_SHAPE, name='image')

# Encodeur : 3 convolutions stridées, le nombre de filtres se resserre pour forcer un vrai goulot
x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(inputs)  # 64x64x32
x = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)       # 32x32x64
latent = layers.Conv2D(8, 3, strides=2, padding='same', activation='relu')(x)   # 16x16x8 (goulot)

# Décodeur : 3 déconvolutions symétriques, sortie sigmoid dans [0, 1]
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(latent)  # 32x32x64
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)        # 64x64x32
outputs = layers.Conv2DTranspose(1, 3, strides=2, padding='same', activation='sigmoid')(x)  # 128x128x1

autoencoder = keras.Model(inputs, outputs, name='autoencodeur_screw')
autoencoder.summary()

Model: "autoencodeur_screw"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 8)      │         4,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 32, 32, 64)     │         4,672 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 64, 64, 32)     │        18,464 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 128, 128, 1)    │           289 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,857 (183.04 KB)

 Trainable params: 46,857 (183.04 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
input_size = np.prod(INPUT_SHAPE)
latent_size = np.prod(latent.shape[1:])
ratio = latent_size / input_size

print(f'Taille entrée       : {input_size} valeurs {INPUT_SHAPE}')
print(f'Taille goulot       : {latent_size} valeurs {tuple(latent.shape[1:])}')
print(f'Ratio goulot/entrée : {ratio:.3f}  (compression ×{1 / ratio:.1f})')

Taille entrée       : 16384 valeurs (128, 128, 1)
Taille goulot       : 2048 valeurs (16, 16, 8)
Ratio goulot/entrée : 0.125  (compression ×8.0)


**Commentaire.** Le modèle compte **46 857 paramètres** (tous entraînables), essentiellement dans les
6 couches de convolution/déconvolution — pas de couche dense, donc pas d'explosion de paramètres malgré
la taille des images (128×128).

Le **goulot d'étranglement** est la sortie de la 3ᵉ convolution : `16×16×8 = 2048` valeurs, contre
`128×128×1 = 16384` valeurs en entrée → **ratio de 0.125** (compression ×8). Ce ratio est volontairement
resserré via le nombre de filtres (32 → 64 → **8**, et non croissant jusqu'au bout) : avec un goulot proche
de la taille d'entrée, le modèle pourrait apprendre une quasi-identité et reconstruire aussi bien les
défauts que les pièces saines, ce qui ferait disparaître le signal d'anomalie recherché.